In [52]:
%matplotlib notebook
import numpy as np
import cvxopt
import cvxopt.glpk

In [53]:
cvxopt.glpk.ilp?

Docstring:
Solves a mixed integer linear program using GLPK.

(status, x) = ilp(c, G, h, A, b, I, B)

PURPOSE
Solves the mixed integer linear programming problem

    minimize    c'*x
    subject to  G*x <= h
                A*x = b
                x[k] is integer for k in I
                x[k] is binary for k in B

ARGUMENTS
c            nx1 dense 'd' matrix with n>=1

G            mxn dense or sparse 'd' matrix with m>=1

h            mx1 dense 'd' matrix

A            pxn dense or sparse 'd' matrix with p>=0

b            px1 dense 'd' matrix

I            set of indices of integer variables

B            set of indices of binary variables

status       if status is 'optimal', 'feasible', or 'undefined',
             a value of x is returned and the status string 
             gives the status of x.  Other possible values of              status are:  'invalid formulation', 
             'infeasible problem', 'LP relaxation is primal 
             infeasible', 'LP relaxation is dual

In [54]:
## To help create the huge matrix

def unravel(i,j,k,size=3):
    n2 = size*size
    assert(i>=0 and i<n2)
    assert(j>=0 and i<n2)
    assert(k>=0 and i<n2)
    
    return(k+ j*n2+ i*n2*n2)


""" not used
def ravel(l,size=3):
    n2=size*size
    assert (l>=0 and l < n2*n2*n2)
    i = l // (n2*n2)
    j = (l % (n2*n2)) // n2
    k = l - i*n2*n2 - j*n2
    return ((i,j,k))
"""

' not used\ndef ravel(l,size=3):\n    n2=size*size\n    assert (l>=0 and l < n2*n2*n2)\n    i = l // (n2*n2)\n    j = (l % (n2*n2)) // n2\n    k = l - i*n2*n2 - j*n2\n    return ((i,j,k))\n'

In [55]:
## create constraint matrix

def createMatrix(size=3):
    n2=size*size
    A=np.zeros((4*n2*n2,n2*n2*n2))
    return (A)

A=createMatrix()
A.shape

(324, 729)

In [56]:

## modifies a global constraint matrix
def sudokuConstraints(size=3):
    global A

    n2 = size*size
    ## line constraints: only one number per line

    r = 0 ## row index 

    for k in range(n2): ## for all numbers
        for j in range(n2): ## for all columns
            for i in range(n2): 
                A[r,unravel(i,j,k)] = 1 ## only one number k on line i
            r += 1
        
    ## column constraints: only one number per column
    for k in range(n2):
        for i in range(n2):
            for j in range(n2):
                A[r,unravel(i,j,k)] = 1
            r += 1
        
    # unicity constraints
    for i in range(n2): ## for all rows
        for j in range(n2): ## for all columns
            for k in range(n2):
                A[r, unravel(i,j,k)] = 1 ## only one number in each position
            r += 1
        
    # sub-square constraints
    for k in range(n2): ## for all numbers
        for i in range(size): ## for all row square
            for j in range(size): ## for all columns squares
                for u in range(size): ## all the lines in the subsquare
                    for v in range(size): ## all the columns in the subsquare
                        A[r, unravel(i*size+u,j*size+v,k)]=1
                r += 1
            
    print("Total number of constraints=",r)
    return;
    
    
def testA(A,r,size=3):
    n2 = size*size
    for n in range(r):
        if (np.sum(A[n,])!=n2):
            print("error on line", n)
            break
    print("All constraints OK")
    return

sudokuConstraints()
testA(A,r)

Total number of constraints= 324
All constraints OK


In [57]:
## add the fixed numbers constraints
Keasy = np.array([
[0,8,0, 9,0,1, 0,5,0],
[0,0,2, 6,8,7, 3,0,0],
[0,0,3, 0,0,0, 6,0,0],
[3,9,0, 0,0,0, 0,6,5],
[6,0,0, 4,7,5, 0,0,3],
[5,7,0, 0,0,0, 0,8,4],
[0,0,9, 0,0,0, 8,0,0],
[0,0,5, 1,2,4, 9,0,0],
[0,4,0, 8,0,3, 0,2,0]])

Khard = np.array(
    [
        [7,0,0, 0,0,0, 4,0,0],
        [0,2,0, 0,7,0, 0,8,0],
        [0,0,3, 0,0,8, 0,0,9],     
        [0,0,0, 5,0,0, 3,0,0],
        [0,6,0, 0,2,0, 0,9,0],
        [0,0,1, 0,0,7, 0,0,6],
        [0,0,0, 3,0,0, 9,0,0],
        [0,3,0, 0,4,0, 0,6,0],
        [0,0,9, 0,0,1, 0,0,5]
    ]
)

def fillfixed(K,size=3):
    global A
    n2=size*size
    for i in range(n2):
        for j in range(n2):
            k = K[i,j]
            if (k>0):
                newrow=np.zeros(n2*n2*n2)
                newrow[unravel(i,j,k-1)]=1
                A=np.vstack((A,newrow))
            
## careful, modifies A
fillfixed(Khard)
            
print("A.shape=", A.shape)

A.shape= (348, 729)


In [65]:
## solving
from cvxopt import matrix

b=matrix(np.ones(A.shape[0])) ## set partition
c=matrix(np.zeros(A.shape[1])) ## zero cost
G=matrix(np.zeros(A.shape))
Gid = matrix(np.identity(A.shape[1]))
hid=matrix(np.ones(A.shape[1]))
h=matrix(np.zeros(A.shape[0]))
#B=set(range(A.shape[1]))
B=set()
Aeq = matrix(A)
(status, solution) = cvxopt.glpk.ilp(c=c,G=Gid,h=hid,A=Aeq,b=b)
print(status)

optimal


In [66]:
## print solution
def printsol(sol):
    sep3="+-----+-----+-----+"
    for i in range(n2):
        if (i%3 == 0):
            print(sep3)
        print("|",end='')
        for j in range(n2):
            for k in range(n2):
                if (sol[unravel(i,j,k)]==1):
                    print(k+1,end='')
            if (j%3 ==2):
                print("|",end='')
            else:
                print(" ",end='')
        print("")
    print(sep3)
        
printsol(solution)
          

+-----+-----+-----+
|1245679 1345789 1235689|134578 14689 1235789|23469 24578 245789|
|1356789 1256 123489|1346789 2356789 124568|12379 124589 123678|
|1246789 23456789 1234678|1235789 1235679 234689|236789 1258 1356789|
+-----+-----+-----+
|3456789 1235679 235678|1245679 13467 4589|1234578 16789 1234589|
|2345789 13468 146789|246789 1234568 357|345678 1235789 1245679|
|134678 1234589 1467|2356789 1234589 345679|134678 234568 25678|
+-----+-----+-----+
|1234568 16789 2356789|134567 1356789 1234789|124689 2379 1236789|
|3569 123578 1345789|234589 45678 236789|4678 235679 13578|
|14589 12389 1234679|24689 123578 1234567|125678 1345689 234589|
+-----+-----+-----+


# computer scientist version

In [ ]:
Ksci = np.array(
    [
        [9,0xF+1,0,0xC+1, 0,0,0,0,         0,0xA+1,0,0, 0,0,0,  7],
        [0,0,0,0xA+1,     0,0,0,0xF+1,     0,0,0,0xB+1, 8,5,0xD+1,0],
        [0xB+1,0,5,0,     0,0,0xD+1,7,     0,8,0,0,     1,0,6,0],
        [2,0,0,0,         0,0,0,1,         4,0,10,3,    0,0,0,0],
        
        [0,0,0,0,         0,2,0xF+1,0xD+1, 0,4,1,0,     0,0xE+1,8,5],
        [0,2,0,7,         0,0,0,0xC+1,     0,0xB+1,0,0, 0xA+1,0,4,0],     
        [0,0xC+1,0,0xD+1, 0,0,7,4,         0,6,0,0,     10,3,0,0],
        [10,0,4,5,        0xE+1,0,3,0,     0,0,7,0xD+1, 0,0,0,0],
        
        [0,0,1, 0,0,7, 0,0,6],
        [0,0,0, 3,0,0, 9,0,0],
        [0,3,0, 0,4,0, 0,6,0],
        [0,0,9, 0,0,1, 0,0,5],
    ]
)

In [43]:
0xB

11

In [51]:
set(range(10))

{0, 1, 2, 3, 4, 5, 6, 7, 8, 9}

In [67]:
np.identity(3)

array([[1., 0., 0.],
       [0., 1., 0.],
       [0., 0., 1.]])